# Retrieval-eval — Data analysis

Load sequential run JSONL into a DataFrame and visualize **hit@k**, **answer_correct**, and token usage by search type, gravity, eval type, and question.

**Setup:** Run this notebook from the `retrieval-eval/` directory (e.g. `jupyter lab .` from `retrieval-eval/`). If no `results/sequential_run_*.jsonl` exist, the fixture `analysis/fixtures/minimal_sequential.jsonl` is used so all cells run.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd() if (Path.cwd() / "run_sequential.py").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from analysis.load_results import load_jsonl, get_default_jsonl_paths

paths = get_default_jsonl_paths(ROOT)
if not paths:
    raise FileNotFoundError("No results/sequential_run_*.jsonl and no analysis/fixtures/minimal_sequential.jsonl")

df = load_jsonl(ROOT, paths)
print(f"Loaded {len(df)} rows from {len(paths)} file(s).")
print(f"Paths: {[str(p) for p in paths]}")

## Data preview

In [ ]:
display(df.head(10))
df.describe(include="all")

In [ ]:
sub = df[~df["is_error"]] if "is_error" in df.columns else df
if not sub.empty:
    print("Non-error runs:", len(sub))
    print("hit@k rate:", sub["hit_at_k"].mean() if "hit_at_k" in sub.columns else "N/A")
    print("answer_correct rate:", sub["answer_correct"].mean() if "answer_correct" in sub.columns else "N/A")

## Visualizations

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
# Heatmap: gravity × search_type → answer_correct rate
sub = df[(~df["is_error"]) & (df["gravity"].notna()) & (df["gravity"] != "n/a")]
if not sub.empty and "search_type" in sub.columns:
    pivot = sub.pivot_table(index="gravity", columns="search_type", values="answer_correct", aggfunc="mean")
    if not pivot.empty:
        fig, ax = plt.subplots(figsize=(max(8, pivot.shape[1] * 1.2), max(4, pivot.shape[0] * 0.6)))
        sns.heatmap(pivot, annot=True, fmt=".2f", cmap="RdYlGn", vmin=0, vmax=1, ax=ax)
        ax.set_title("Answer correct rate: gravity × search_type")
        plt.tight_layout()
        plt.show()
else:
    print("Skipped (no gravity/search_type variety or no non-error rows).")

In [ ]:
# Bars: eval_type — hit@k vs answer_correct
sub = df[~df["is_error"]]
if not sub.empty and "eval_type" in sub.columns:
    g = sub.groupby("eval_type", dropna=False).agg(
        hit_at_k=("hit_at_k", "mean"),
        answer_correct=("answer_correct", "mean"),
    ).reset_index()
    fig, ax = plt.subplots(figsize=(10, 5))
    x = range(len(g))
    w = 0.35
    ax.bar([i - w/2 for i in x], g["hit_at_k"], width=w, label="hit@k")
    ax.bar([i + w/2 for i in x], g["answer_correct"], width=w, label="answer_correct")
    ax.set_xticks(list(x))
    ax.set_xticklabels(g["eval_type"], rotation=25, ha="right")
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.set_title("Retrieval vs answer accuracy by eval_type")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped (no eval_type or no non-error rows).")

In [ ]:
# Bars: thinking vs direct
sub = df[~df["is_error"]].copy()
if not sub.empty and "thinking" in sub.columns:
    sub["thinking_label"] = sub["thinking"].map({True: "thinking", False: "direct"})
    g = sub.groupby("thinking_label", dropna=False).agg(
        hit_at_k=("hit_at_k", "mean"),
        answer_correct=("answer_correct", "mean"),
    ).reset_index()
    fig, ax = plt.subplots(figsize=(6, 4))
    x = range(len(g))
    w = 0.35
    ax.bar([i - w/2 for i in x], g["hit_at_k"], width=w, label="hit@k")
    ax.bar([i + w/2 for i in x], g["answer_correct"], width=w, label="answer_correct")
    ax.set_xticks(list(x))
    ax.set_xticklabels(g["thinking_label"])
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.set_title("Thinking vs direct")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped.")

In [ ]:
# Line: per-question difficulty (answer_correct rate by eval_type and question_index)
sub = df[~df["is_error"]]
if not sub.empty and "question_index" in sub.columns and "eval_type" in sub.columns:
    g = sub.groupby(["eval_type", "question_index"], dropna=False).agg(
        answer_correct=("answer_correct", "mean"),
        n=("answer_correct", "count"),
    ).reset_index()
    fig, ax = plt.subplots(figsize=(10, 5))
    for et in g["eval_type"].dropna().unique():
        part = g[g["eval_type"] == et]
        ax.plot(part["question_index"], part["answer_correct"], marker="o", label=str(et))
    ax.set_xlabel("question_index")
    ax.set_ylabel("answer_correct rate")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(title="eval_type", bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.set_title("Per-question difficulty (mean answer_correct)")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped.")

In [ ]:
# Scatter: token use vs correctness
sub = df[(~df["is_error"]) & (df["total_tokens"].notna()) & (df["total_tokens"] > 0)]
if len(sub) >= 5:
    fig, ax = plt.subplots(figsize=(8, 5))
    ok = sub[sub["answer_correct"] == True]
    bad = sub[sub["answer_correct"] == False]
    ax.scatter(ok["total_tokens"], [1] * len(ok), alpha=0.4, label="correct", s=20)
    ax.scatter(bad["total_tokens"], [0] * len(bad), alpha=0.4, label="incorrect", s=20)
    ax.set_xlabel("total_tokens")
    ax.set_ylabel("answer_correct (0/1)")
    ax.set_title("Token use vs correctness")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Skipped (need at least 5 non-error rows with total_tokens).")

In [ ]:
# Bars: compare runs (by source_file)
if "source_file" in df.columns and df["source_file"].nunique() >= 2:
    sub = df[~df["is_error"]]
    g = sub.groupby("source_file").agg(
        answer_correct=("answer_correct", "mean"),
        n=("answer_correct", "count"),
    ).reset_index()
    fig, ax = plt.subplots(figsize=(max(8, len(g) * 0.8), 4))
    sns.barplot(data=g, x="source_file", y="answer_correct", ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_title("answer_correct rate by result file (compare runs)")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped (single file or no source_file).")

## Summary tables

In [ ]:
sub = df[~df["is_error"]] if "is_error" in df.columns else df
if not sub.empty:
    for dim in ["search_type", "eval_type", "gravity"]:
        if dim in sub.columns:
            t = sub.groupby(dim, dropna=False).agg(
                n=("answer_correct", "count"),
                hit_at_k=("hit_at_k", "mean"),
                answer_correct=("answer_correct", "mean"),
            ).reset_index()
            print(f"By {dim}:")
            display(t)
            print()